# Zadanie 3 — pandas: rozpoznanie i czyszczenie danych

**Czas:** 30–40 minut  
**Poziom:** podstawowy/średni  
**Cel:** przygotować niewielką tabelę budynków do analizy bez ukrywania przyjętych decyzji.

Dane zawierają celowe problemy: duplikat, brak wyniku, błędny metraż i rok zapisany tekstem. Nie wczytujesz żadnego pliku; tabela powstaje w pierwszej komórce.


> **WERSJA DLA PROWADZĄCEGO — kompletne rozwiązanie.**
>
> Uruchom komórki od góry. Kod, wyniki kontrolne i interpretacje są częścią odpowiedzi. Dane są syntetyczne.


In [ ]:
import numpy as np
import pandas as pd

raw = pd.DataFrame([
    {"id": "A01", "area_m2": 40, "year": "2008", "kwh": 180},
    {"id": "A02", "area_m2": 65, "year": "2015", "kwh": 293},
    {"id": "A03", "area_m2": 82, "year": "1999", "kwh": 369},
    {"id": "A04", "area_m2": 55, "year": "2020", "kwh": np.nan},
    {"id": "A05", "area_m2": 120, "year": "1988", "kwh": 620},
    {"id": "A06", "area_m2": 73, "year": "2012", "kwh": 340},
    {"id": "A06", "area_m2": 73, "year": "2012", "kwh": 340},
    {"id": "A07", "area_m2": -12, "year": "2018", "kwh": 210},
    {"id": "A08", "area_m2": 90, "year": "brak", "kwh": 410},
])

raw


## A. Diagnoza

1. Wyświetl `head()`, `info()` i liczbę braków w każdej kolumnie.
2. Sprawdź liczbę powtórzeń identyfikatora `id`.
3. Wypisz wiersze z metrażem niedodatnim oraz wiersze z brakującą wartością `kwh`.
4. W komórce Markdown napisz, dlaczego warto obejrzeć problematyczne wiersze **przed** ich usunięciem.


In [ ]:
print("Pierwsze rekordy:")
print(raw.head())
print("\nInformacje o tabeli:")
raw.info()
print("\nLiczba braków:")
print(raw.isna().sum())
print("Duplikaty id:", int(raw.duplicated(subset="id").sum()))
print("\nPowtórzone identyfikatory:")
print(raw.loc[raw.duplicated(subset="id", keep=False)])
print("\nNiedodatni metraż:")
print(raw.loc[raw["area_m2"] <= 0])
print("\nBrak celu kWh:")
print(raw.loc[raw["kwh"].isna()])


**Dlaczego oglądamy problematyczne wiersze?** Duplikat może być poprawnym ponownym pomiarem, a brak wyniku może mieć przyczynę systemową. W tym ćwiczeniu reguły czyszczenia są narzucone, lecz w realnym projekcie decyzję podejmujemy po zbadaniu znaczenia rekordów.


## B. Ustalona polityka czyszczenia

W tej kolejności:

5. Utwórz kopię `clean = raw.copy()`. Usuń powtórzony identyfikator, zachowując pierwszy rekord.
6. Zamień `year` na wartości liczbowe przez `pd.to_numeric(..., errors="coerce")`.
7. Zamień niedodatni metraż na brak (`NaN`).
8. Usuń obserwację z brakującym `kwh`, ponieważ `kwh` będzie wartością docelową.
9. Uzupełnij brakujący metraż medianą **z pozostałych poprawnych metraży**, a brakujący rok medianą **z pozostałych poprawnych lat**.

Zapisz w zmiennych `area_median` i `year_median` wartości użyte do uzupełnienia. Po każdym ważniejszym kroku możesz wyświetlić liczbę wierszy.


In [ ]:
clean = raw.copy()
clean = clean.drop_duplicates(subset="id", keep="first")
print("Po usunięciu duplikatu:", len(clean))

clean["year"] = pd.to_numeric(clean["year"], errors="coerce")
clean.loc[clean["area_m2"] <= 0, "area_m2"] = np.nan

clean = clean.dropna(subset=["kwh"]).copy()
print("Po usunięciu rekordu bez kWh:", len(clean))

area_median = float(clean["area_m2"].median())
year_median = float(clean["year"].median())
clean["area_m2"] = clean["area_m2"].fillna(area_median)
clean["year"] = clean["year"].fillna(year_median)

print("Mediana metrażu:", area_median)
print("Mediana roku:", year_median)
print("Braki po czyszczeniu:")
print(clean.isna().sum())


## C. Nowe cechy i interpretacja

10. Dodaj kolumnę `age_2026 = 2026 - year` oraz `kwh_per_m2 = kwh / area_m2`.
11. Uporządkuj wynik malejąco według `kwh_per_m2`.
12. Wypisz identyfikatory budynków z intensywnością powyżej **5 kWh/m²**.
13. Napisz trzy zdania: które rekordy poprawiono, który usunięto i jakie ograniczenie ma uzupełnianie medianą w małym zbiorze.

**Uwaga metodologiczna:** jeśli przygotowujesz cechy do modelu ML, medianę wyznaczaj na treningu i stosuj do testu. Tutaj ćwiczymy czyszczenie jednej tabeli, bez podziału na trening i test.


In [ ]:
clean["age_2026"] = 2026 - clean["year"]
clean["kwh_per_m2"] = clean["kwh"] / clean["area_m2"]

ranking = clean.sort_values("kwh_per_m2", ascending=False)
high_intensity = ranking.loc[ranking["kwh_per_m2"] > 5, "id"]

print(ranking[["id", "area_m2", "year", "kwh", "age_2026", "kwh_per_m2"]])
print("Budynki powyżej 5 kWh/m²:", high_intensity.tolist())


### Interpretacja prowadzącego

1. Usunięto powtórzony rekord A06; niedodatni metraż A07 zastąpiono medianą **77,5 m²**, a nierozpoznany rok A08 medianą **2010**.
2. Usunięto A04, ponieważ brakowało `kwh`, czyli wartości docelowej w tym ćwiczeniu.
3. Uzupełnienie medianą pozwala zachować rekord, ale przy tak małym zbiorze silnie zależy od kilku dostępnych wartości i może ukryć ważną przyczynę braku. W pełnym procesie ML medianę wyznaczamy wyłącznie na treningu.


## Samokontrola

Uruchom dopiero po zapisaniu wszystkich decyzji.


In [ ]:
assert len(clean) == 7
assert clean["id"].is_unique
assert not clean[["area_m2", "year", "kwh"]].isna().any().any()
assert np.isclose(area_median, 77.5)
assert np.isclose(year_median, 2010.0)
assert np.isclose(clean.loc[clean["id"] == "A07", "area_m2"].iloc[0], 77.5)
assert np.isclose(clean.loc[clean["id"] == "A08", "year"].iloc[0], 2010.0)
assert "A05" in clean.loc[clean["kwh_per_m2"] > 5, "id"].tolist()
print("Kontrola danych zakończona pomyślnie.")


## Wyzwanie dodatkowe

Porównaj średnią `kwh_per_m2` przed i po wyłączeniu rekordu A05. Czy ten jeden obiekt silnie wpływa na średnią?


## Rozwiązanie wyzwania dodatkowego

Porównujemy średnią intensywność w całej wyczyszczonej tabeli i po pominięciu A05. To badanie wpływu konkretnej obserwacji, a nie reguła usuwania jej z projektu.


In [ ]:
mean_all = float(clean["kwh_per_m2"].mean())
mean_without_A05 = float(clean.loc[clean["id"] != "A05", "kwh_per_m2"].mean())
print(f"Średnia ze wszystkimi: {mean_all:.3f} kWh/m²")
print(f"Średnia bez A05: {mean_without_A05:.3f} kWh/m²")
print(f"Różnica: {mean_all - mean_without_A05:.3f} kWh/m²")


**Wniosek:** A05 podwyższa średnią intensywność, ale interpretację wpływu należy odnieść do skali i znaczenia różnicy. Warto obejrzeć również medianę i rozkład, zamiast rozstrzygać na podstawie samej średniej.
